# День 6 — Валидация, overfitting и подбор параметров

## Цель
Увидеть, что модель может хорошо помнить **train**, но плохо работать на новых данных, и научиться проверять её устойчивее через **cross-validation**.

## Overfitting / underfitting

| | Что происходит | Как выглядит |
|---|----------------|--------------|
| **Overfitting** | модель слишком хорошо подстроилась под train | train score высокий, test score заметно ниже |
| **Underfitting** | модель слишком простая | и train, и test score низкие |
| **Хорошо** | модель обобщает | train и test близки, оба приемлемые |

В day 5 дерево без ограничений на Diabetes было слабее линейной модели — типичный намёк на overfitting или слишком «шумную» подгонку.

**Практический приём дня:** для каждого `max_depth` сравниваем **train score** и **test score**. Разрыв train ≫ test → подозрение на overfitting.

## Cross-validation (CV)

Один `train_test_split` — это **одна** случайная проверка. Результат может «повезти» или «не повезти» из‑за конкретного разбиения.

**K-fold CV** делит данные на **K** частей (фолдов). K раз:
1. учимся на K−1 фолдах;
2. проверяем на оставшемся фолде.

В sklearn:

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
scores.mean()  # средний результат
scores.std()   # насколько нестабилен
```

- **mean** — типичное качество
- **std** — разброс между фолдами (высокий std → результат зависит от разбиения)

CV отвечает: «насколько результат зависит от конкретного split?»

## Гиперпараметры

**Гиперпараметр** — настройка модели *до* обучения (не выучивается из данных сама).

Параметры модели (веса, пороги в листьях) — модель подбирает на `fit`.  
Гиперпараметры (`max_depth`, `min_samples_leaf`) — задаём мы.

| Параметр | Смысл | Крайности |
|----------|--------|-----------|
| `max_depth` | максимальная глубина дерева | `1` — очень простое; `None` — без ограничения |
| `min_samples_leaf` | минимум объектов в листе | больше → дерево проще |

Сегодня крутим **`max_depth = 1, 2, 3, 5, None`** и смотрим, где underfitting, где overfitting.

## График max_depth → score

Line plot помогает увидеть компромисс:

- слева (маленький `max_depth`) — часто underfitting;
- справа (`None` / очень глубокое) — часто overfitting;
- в середине — зона, где train и test ближе.

Удобно рисовать **две линии**: train score и test (или CV) score.

## Задания

Датасет: **Diabetes** + `DecisionTreeRegressor` (продолжение day 5).

1. Загрузить `load_diabetes()`, собрать X/y, сделать train/test split.
2. Для `max_depth = 1, 2, 3, 5, None` обучить дерево и сравнить **train** vs **test** score (R²).
3. Для тех же глубин сделать `cross_val_score` (cv=5) → mean / std.
4. Построить line plot: `max_depth` → train / test (или CV) score → `week-3/figures/`.
5. Написать выводы: underfitting / overfitting / какой `max_depth` выбрать.


In [ ]:
# 1. Загрузка Diabetes и train/test split (как в day 5)

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

dataset = load_diabetes(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Признаки:", list(X.columns))
print("y (первые 5):", y.head().tolist())


In [ ]:
# 2. max_depth = 1, 2, 3, 5, None — train vs test R²

import pandas as pd
from sklearn.tree import DecisionTreeRegressor

depths = [1, 2, 3, 5, None]
rows = []

for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    # print(f"model: {model}")
    # print("===================")

    train_r2 = model.score(X_train, y_train)  # R² на train
    test_r2 = model.score(X_test, y_test)    # R² на test

    # print(f"train_r2: {train_r2}")
    # print(f"test_r2: {test_r2}")
    # print("===================")

    rows.append({
        "max_depth": depth if depth is not None else "None",
        "train_R2": round(train_r2, 3),
        "test_R2": round(test_r2, 3),
        "gap": round(train_r2 - test_r2, 3),  # train − test
    })

    # print(f"rows: {rows}")
    # print("===================")

results = pd.DataFrame(rows)
results


In [ ]:
# 3. cross_val_score (cv=5) для тех же max_depth

from sklearn.model_selection import cross_val_score

cv_rows = []

for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    # scoring по умолчанию для регрессора = R²
    scores = cross_val_score(model, X, y, cv=5)

    cv_rows.append({
        "max_depth": depth if depth is not None else "None",
        "cv_mean": round(scores.mean(), 3),
        "cv_std": round(scores.std(), 3),
        "scores": [round(s, 3) for s in scores],
    })

cv_results = pd.DataFrame(cv_rows)
cv_results


In [ ]:
# 4. Line plot: max_depth → train / test / CV R²

from pathlib import Path
import matplotlib.pyplot as plt

x_labels = results["max_depth"].astype(str).tolist()
x = range(len(x_labels))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, results["train_R2"], marker="o", label="train R²")
ax.plot(x, results["test_R2"], marker="o", label="test R²")
ax.plot(x, cv_results["cv_mean"], marker="o", label="CV mean R²")

ax.set_xticks(list(x))
ax.set_xticklabels(x_labels)
ax.set_xlabel("max_depth")
ax.set_ylabel("R²")
ax.set_title("DecisionTreeRegressor: max_depth → score (Diabetes)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

figures_dir = Path("week-3/figures")
if not figures_dir.exists():
    figures_dir = Path("../figures")  # если запуск из week-3/notebooks/
figures_dir.mkdir(parents=True, exist_ok=True)

fig_path = figures_dir / "06_max_depth_vs_score.png"
fig.savefig(fig_path, dpi=120)
print("Saved:", fig_path.resolve())
plt.show()


## Выводы

1. **Underfitting** — при `max_depth=1`: и train R² (~0.30), и test/CV (~0.13–0.17) низкие — дерево слишком простое.
2. **Overfitting** — особенно при `max_depth=None`: train R² = **1.0**, test ≈ **0.06**, CV mean ≈ **−0.15**. Модель запомнила train и плохо обобщает.
3. Лучший компромисс сейчас — **`max_depth=2`** (рядом `3`): выше CV mean (~0.33), разрыв train−test умеренный. `5` уже раздувает gap без выигрыша на CV.
4. Один train/test split полезен, но **CV** показал устойчивее: `None` стабильно плох на всех фолдах, а пик качества действительно около глубины 2–3.
5. Связь с day 5: unrestricted дерево на Diabetes слабое не «случайно» — без ограничения глубины оно переобучается.
6. Гиперпараметр `max_depth` реально меняет поведение модели: ограничение сложности помогает обобщению.
7. Даже лучшее дерево здесь слабее LinearRegression из day 5 (R² ~0.45) — для этого датасета линейная модель всё ещё сильнее.
8. Дальше: day 7 — собрать мини-проект с baseline, моделями, метриками и понятными выводами.


---

# Day 6 — Validation, overfitting, and parameter tuning

## Goal
See that a model can memorize **train** but fail on new data, and learn to check it more robustly with **cross-validation**.

## Overfitting / underfitting

| | What happens | How it looks |
|---|--------------|--------------|
| **Overfitting** | model fits train too tightly | high train score, clearly lower test score |
| **Underfitting** | model too simple | both train and test scores are low |
| **Good fit** | model generalizes | train and test are close and acceptable |

In day 5, an unrestricted tree on Diabetes was weaker than linear regression — a hint of overfitting / noisy fit.

**Practical check:** for each `max_depth`, compare **train score** vs **test score**. A large gap (train ≫ test) suggests overfitting.

## Cross-validation (CV)

One `train_test_split` is a **single** random check. Results can look lucky or unlucky depending on the split.

**K-fold CV** splits data into **K** folds. K times:
1. train on K−1 folds;
2. evaluate on the held-out fold.

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
scores.mean()
scores.std()
```

- **mean** — typical quality
- **std** — instability across folds

CV answers: «how much does the result depend on this particular split?»

## Hyperparameters

A **hyperparameter** is set *before* training (not learned from data).

| Parameter | Meaning | Extremes |
|-----------|---------|----------|
| `max_depth` | max tree depth | `1` — very simple; `None` — unrestricted |
| `min_samples_leaf` | min samples in a leaf | larger → simpler tree |

Today we sweep **`max_depth = 1, 2, 3, 5, None`** and look for underfitting vs overfitting.

## Plot: max_depth → score

A line plot shows the trade-off:

- left (small `max_depth`) — often underfitting;
- right (`None` / very deep) — often overfitting;
- middle — where train and test are closer.

Useful to plot **two lines**: train score and test (or CV) score.

## Tasks

Dataset: **Diabetes** + `DecisionTreeRegressor` (continues day 5).

1. Load `load_diabetes()`, build X/y, train/test split.
2. For `max_depth = 1, 2, 3, 5, None`, train a tree and compare **train** vs **test** score (R²).
3. For the same depths, run `cross_val_score` (cv=5) → mean / std.
4. Line plot: `max_depth` → train / test (or CV) score → `week-3/figures/`.
5. Write conclusions: underfitting / overfitting / which `max_depth` to pick.


## Conclusions

1. **Underfitting** at `max_depth=1`: both train R² (~0.30) and test/CV (~0.13–0.17) are low — the tree is too simple.
2. **Overfitting** especially at `max_depth=None`: train R² = **1.0**, test ≈ **0.06**, CV mean ≈ **−0.15**. The model memorized train and fails to generalize.
3. Best compromise now: **`max_depth=2`** (close to `3`) — highest CV mean (~0.33) with a moderate train−test gap. Depth `5` widens the gap without CV gains.
4. A single train/test split helps, but **CV** is more robust: `None` is consistently bad across folds; the quality peak is around depth 2–3.
5. Link to day 5: the unrestricted Diabetes tree was weak for a reason — unlimited depth overfits.
6. The hyperparameter `max_depth` truly changes behavior: limiting complexity improves generalization.
7. Even the best tree here is weaker than LinearRegression from day 5 (R² ~0.45) — linear still wins on this dataset.
8. Next: day 7 — assemble a mini-project with baseline, models, metrics, and clear conclusions.
